In [1]:
from src.service.template_service import Template_ETL
from src.core.settings import settings
from src.core.template import (
    column_check_oa, column_promotion_plan, column_update_so, column_missing_ou, 
    column_so_calendar, column_purchase, column_po_commitment, column_supplier_schedule, 
    column_ag, column_dc, column_de, report, column_add_attribute_marketing, column_sale
)
from src.core.constants import (
    VAT, required_stage2
)
from typing import Optional, List

import pandas as pd
import re

In [2]:
etl = Template_ETL(
    settings.path_src,
    settings.path_mdt,
    settings.path_sitegroup,
    settings.path_plan
)

In [3]:
etl._load_mdt()
etl._load_sitegroup()
etl._load_network()
etl._load_src()
etl._pipeline()
etl._load_plan()

D:\Code\ai-agents-data-portal\gold-promo-agents\src\service\template_service.py:142: ParserWarning: Skipping line 125387: '|' expected after '"'

  data = pd.read_csv(
D:\Code\ai-agents-data-portal\gold-promo-agents\src\service\template_service.py:142: ParserWarning: Skipping line 160229: '|' expected after '"'

  data = pd.read_csv(
D:\Code\ai-agents-data-portal\gold-promo-agents\src\service\template_service.py:142: ParserWarning: Skipping line 319518: '|' expected after '"'

  data = pd.read_csv(
D:\Code\ai-agents-data-portal\gold-promo-agents\src\service\template_service.py:142: ParserWarning: Skipping line 454386: '|' expected after '"'

  data = pd.read_csv(
D:\Code\ai-agents-data-portal\gold-promo-agents\src\service\template_service.py:142: ParserWarning: Skipping line 578971: '|' expected after '"'

  data = pd.read_csv(
D:\Code\ai-agents-data-portal\gold-promo-agents\src\service\template_service.py:347: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the defa

In [4]:
etl._load_src_listoff()
etl._pipeline2()

In [5]:
class ContractMixin:
    """Logic ghép mã CONTRACT từ CONTRACT/SITE/SUPPLIER."""

    def get_contract(self, row):
        contract = row["CONTRACT"]
        site = row["SITE"]
        supplier = row["SUPPLIER"]

        if len(contract) == 8:
            return contract

        if supplier in self.nw.get("wh8"):
            return f"{contract}0{supplier}"

        if len(site) == 3:
            return f"{contract}0{site}"
        elif len(site) == 4:
            return f"{contract}{site}"


class StageMixin:
    """Các bước xử lý DataFrame dùng chung cho mọi template."""

    @staticmethod
    def reset_no(data):
        data = data.reset_index(drop=True)
        data.insert(0, "NO", range(1, len(data) + 1))
        return data

    def fast_stage(self, data, have_no=False):
        data = data.drop_duplicates()
        # data = data.dropna()
        data = data.reset_index(drop=True)
        if have_no:
            data = self.reset_no(data)
        return data


class CalendarMixin:
    """Logic tính ORDER DATE / DELIVERY DATE theo DELIVERY TYPE."""

    def _get_calendar(self, x):
        delivery_type = x["DELIVERY TYPE"]

        if delivery_type == "CROSS-DOCKING":
            order_cols = [
                "CROSS-DOCKING | ORDER DATE 1",
                "CROSS-DOCKING | ORDER DATE 2",
                "CROSS-DOCKING | ORDER DATE 3",
            ]
        elif delivery_type == "DIRECT":
            order_cols = [
                "DIRECT | ORDER DATE 1",
                "DIRECT | ORDER DATE 2",
                "DIRECT | ORDER DATE 3",
            ]
        else:
            order_cols = [
                "VINAMILK | ORDER DATE 1",
                "VINAMILK | ORDER DATE 2",
                "VINAMILK | ORDER DATE 3",
            ]

        delivery_cols = [
            "DELIVERY DATE 1",
            "DELIVERY DATE 2",
            "DELIVERY DATE 3",
        ]

        order_date_all = self.plan[order_cols].iloc[0].tolist()
        delivery_date_all = self.plan[delivery_cols].iloc[0].tolist()

        deli_values = x["%DELI"]

        order_date = []
        delivery_date = []

        for i, value in enumerate(deli_values):
            has_value = pd.notna(value) and str(value).strip() != ""

            if has_value:
                order_date.append(order_date_all[i])
                delivery_date.append(delivery_date_all[i])

        return pd.Series(
            {
                "ORDER DATE": order_date,
                "DELIVERY DATE": delivery_date,
            }
        )


class AllocationMixin:
    """Xây self.allocation: map KEY -> VALUE để tra commitment quantity."""

    def _get_allocation(self) -> "AllocationMixin":
        data = self.src

        site_column = [i for i in data.columns if i in self.nw["store"]]

        allocation = data[
            ["GOLD CODE", "LV", "SUPPLIER CODE", "PURCHASE NETWORK EXPANDED", *site_column]
        ]

        allocation["PURCHASE NETWORK EXPANDED"] = allocation[
            "PURCHASE NETWORK EXPANDED"
        ].str.split(";")

        allocation = allocation.explode("PURCHASE NETWORK EXPANDED")

        allocation["KEY"] = (
            allocation["GOLD CODE"].astype(str)
            + "-"
            + allocation["LV"].astype(str)
            + "-"
            + allocation["SUPPLIER CODE"].astype(str)
            + "-"
            + allocation["PURCHASE NETWORK EXPANDED"]
        )

        allocation["VALUE"] = allocation.apply(
            lambda x: x[x["PURCHASE NETWORK EXPANDED"]],
            axis=1,
        )

        network_dict = dict(zip(allocation["KEY"], allocation["VALUE"]))

        self.allocation = network_dict

        return self


class AttributeMapMixin:
    """Map MEDIUM (free text) -> category chuẩn hoá bằng regex."""

    CATEGORY_RULES: Optional[List] = [
        (
            re.compile(r"(?i)\b(front\s*page|back\s*page|unbeat)\b"),
            "HERO",
        ),
        (
            re.compile(
                r"(?i)\b(cata|catalog(?:ue)?|fair|member\s*price|banner|exclusive\s*pack|family|other)\b"
            ),
            "CATA",
        ),
        (
            re.compile(r"(?i)\b(comple(?:mentary)?|comple)\b"),
            "COMPLE",
        ),
        (
            re.compile(r"(?i)\bbuy\s*more\s*save\s*more\b"),
            "STAR",
        ),
    ]

    def attribute_map(self, text: str) -> str:
        if not text:
            return text

        text = str(text).strip()

        for pattern, value in self.CATEGORY_RULES:
            if pattern.search(text):
                return value

        return text


# --------------------------------------------------------------------------- #
# Base: thuộc tính chung + __init__
# --------------------------------------------------------------------------- #
class BaseTemplate(ContractMixin, StageMixin):
    """
    Chứa __init__ và toàn bộ state dùng chung. Mọi mixin template khác đều
    giả định đã có sẵn: self.src, self.nw, self.plan, self.cata,
    self.cata_description, self.cata_period.
    """

    def __init__(self, etl: "Template_ETL"):
        self.src = etl.src
        self.nw = etl.dict_network
        self.plan = etl.plan
        self.cata = etl.cata
        self.cata_description = etl.cata_description
        self.cata_period = etl.cata_period

        self.allocation: dict = dict()

        self.template_check_oa: Optional[pd.DataFrame] = None
        self.template_promotion_plan: Optional[pd.DataFrame] = None
        self.template_update_so: Optional[pd.DataFrame] = None
        self.template_missing_ou: Optional[pd.DataFrame] = None
        self.template_so_calendar: Optional[pd.DataFrame] = None
        self.template_purchase: Optional[pd.DataFrame] = None
        self.template_po_commitment: Optional[pd.DataFrame] = None
        self.template_supplier_schedule: Optional[pd.DataFrame] = None
        self.template_add_attribute_marketing: Optional[pd.DataFrame] = None
        
class CheckOAMixin:
    def _create_check_oa(self) -> "CheckOAMixin":
        data = self.src

        template_check_oa = {
            column_check_oa[0]: data["GOLD CODE"],
            column_check_oa[1]: data["LV"],
            column_check_oa[2]: data["LU"],
            column_check_oa[3]: data["PURCHASE NETWORK EXPANDED"],
            column_check_oa[4]: data["SUPPLIER CODE"],
            column_check_oa[5]: "1",
            column_check_oa[6]: data["COMMERCIAL CONTRACT"],
            column_check_oa[7]: data["PP START DATE"].dt.strftime("%d/%m/%Y"),
            column_check_oa[8]: data["PP END DATE"].dt.strftime("%d/%m/%Y"),
        }

        template_check_oa = pd.DataFrame(template_check_oa)

        template_check_oa["SITE"] = template_check_oa["SITE"].str.split(";")

        template_check_oa = template_check_oa.explode("SITE")

        template_check_oa["CONTRACT"] = template_check_oa.apply(self.get_contract, axis=1)

        template_check_oa = template_check_oa[column_check_oa]

        template_check_oa = self.fast_stage(template_check_oa, have_no=True)

        self.template_check_oa = template_check_oa

        return self


class PromotionPlanMixin:
    def _create_promotion_plan(self) -> "PromotionPlanMixin":
        data = self.src

        template_promotion_plan = {
            column_promotion_plan[0]: data["SO"],
            column_promotion_plan[1]: f"{self.cata} {self.cata_description} ({self.cata_period})",
            column_promotion_plan[2]: f"{self.cata}D",
            column_promotion_plan[3]: data["SITE GROUP"],
            column_promotion_plan[4]: "1",
            column_promotion_plan[5]: self.plan["CATALOGUE START"].dt.strftime("%d/%m/%Y")[0],
            column_promotion_plan[6]: self.plan["CATALOGUE END"].dt.strftime("%d/%m/%Y")[0],
            column_promotion_plan[7]: self.plan["GLOBAL PERIOD START"][0],
            column_promotion_plan[8]: self.plan["GLOBAL PERIOD END"][0],
            column_promotion_plan[9]: self.plan["SHOP ACTIVATION"][0],
            column_promotion_plan[10]: self.plan["COMMITMENT DEADLINE"][0],
            column_promotion_plan[11]: self.plan["COMMITMENT CLOSING"][0],
            column_promotion_plan[12]: self.plan["ORDER WAREHOUSE START"][0],
            column_promotion_plan[13]: self.plan["ORDER WAREHOUSE END"][0],
        }

        template_promotion_plan = pd.DataFrame(template_promotion_plan)

        template_promotion_plan = template_promotion_plan[column_promotion_plan]

        template_promotion_plan = self.fast_stage(template_promotion_plan, have_no=False)

        self.template_promotion_plan = template_promotion_plan

        return self


class UpdateSOMixin:
    def _create_update_so(self) -> "UpdateSOMixin":
        data = self.src

        template_update_so = {
            column_update_so[0]: "1",
            column_update_so[1]: data["SO"],
            column_update_so[2]: data["GOLD CODE"],
            column_update_so[3]: data["LV"],
            column_update_so[4]: data["LU"],
        }

        template_update_so = pd.DataFrame(template_update_so)

        template_update_so = template_update_so[column_update_so]

        template_update_so = self.fast_stage(template_update_so, have_no=True)

        self.template_update_so = template_update_so

        return self


class MissingOUMixin:
    def _create_missing_ou(self) -> "MissingOUMixin":
        data = self.src

        template_missing_ou = {
            column_missing_ou[0]: data["SO"],
            column_missing_ou[1]: data["PURCHASE NETWORK EXPANDED"],
            column_missing_ou[2]: data["GOLD CODE"],
            column_missing_ou[3]: data["LV"],
            column_missing_ou[4]: data["LU"],
            column_missing_ou[5]: data["SUPPLIER CODE"],
            column_missing_ou[6]: "1",
            column_missing_ou[7]: data["COMMERCIAL CONTRACT"],
            column_missing_ou[8]: "1",
            column_missing_ou[9]: "10",
        }

        template_missing_ou = pd.DataFrame(template_missing_ou)

        template_missing_ou["SITE"] = template_missing_ou["SITE"].str.split(";")

        template_missing_ou = template_missing_ou.explode("SITE")

        template_missing_ou["CONTRACT"] = template_missing_ou.apply(self.get_contract, axis=1)

        template_missing_ou = template_missing_ou[column_missing_ou]

        template_missing_ou = self.fast_stage(template_missing_ou, have_no=True)

        self.template_missing_ou = template_missing_ou

        return self


class SOCalendarMixin(CalendarMixin):
    def _create_so_calendar(self) -> "SOCalendarMixin":
        data = self.src

        template_so_calendar = {
            column_so_calendar[0]: data["SO"],
            column_so_calendar[1]: data["PURCHASE NETWORK EXPANDED"],
            column_so_calendar[2]: data["SUPPLIER CODE"],
            column_so_calendar[3]: data["COMMERCIAL CONTRACT"],
            column_so_calendar[4]: "1",
            column_so_calendar[5]: data["GOLD CODE"],
            column_so_calendar[6]: data["LV"],
            column_so_calendar[7]: data["LU"],
            column_so_calendar[-3]: data[
                ["% DELIVERY 1", "% DELIVERY 2", "% DELIVERY 3"]
            ].values.tolist(),
            column_so_calendar[-2]: "10",
            column_so_calendar[-1]: "",
            "DELIVERY TYPE": data["DELIVERY TYPE"],
        }

        template_so_calendar = pd.DataFrame(template_so_calendar)

        template_so_calendar[["ORDER DATE", "DELIVERY DATE"]] = template_so_calendar.apply(
            self._get_calendar, axis=1
        )

        template_so_calendar.rename(columns={"%DELI": "PCT WEIGHT"}, inplace=True)

        template_so_calendar = template_so_calendar.explode(
            ["PCT WEIGHT", "ORDER DATE", "DELIVERY DATE"]
        )

        template_so_calendar["SITE"] = template_so_calendar["SITE"].str.split(";")

        template_so_calendar = template_so_calendar.explode("SITE")

        template_so_calendar["CONTRACT"] = template_so_calendar.apply(self.get_contract, axis=1)

        template_so_calendar = template_so_calendar[column_so_calendar]

        template_so_calendar = self.fast_stage(template_so_calendar, have_no=True)

        self.template_so_calendar = template_so_calendar

        return self


class PurchaseMixin:
    def _create_purchase(self) -> "PurchaseMixin":
        data = self.src

        template_purchase = {
            column_purchase[0]: data["GOLD CODE"],
            column_purchase[1]: data["LV"],
            column_purchase[2]: "1",
            column_purchase[3]: data["NORMAL PURCHASE PRICE"],
            column_purchase[4]: data["PURCHASE NETWORK EXPANDED"],
            column_purchase[5]: data["PP START DATE"].dt.strftime("%d/%m/%Y"),
            column_purchase[6]: data["PP END DATE"].dt.strftime("%d/%m/%Y"),
            column_purchase[7]: data["COMMERCIAL CONTRACT"],
            column_purchase[8]: data["PURCHASE VAT"].map(VAT),
            column_purchase[9]: data["SUPPLIER CODE"],
            column_purchase[10]: "0",
        }

        template_purchase = pd.DataFrame(template_purchase)

        template_purchase["SITE"] = template_purchase["SITE"].str.split(";")

        template_purchase = template_purchase.explode("SITE")

        template_purchase["CONTRACT"] = template_purchase.apply(self.get_contract, axis=1)

        template_purchase = template_purchase[column_purchase]

        template_purchase = self.fast_stage(template_purchase, have_no=True)

        self.template_purchase = template_purchase

        return self


class POCommitmentMixin(AllocationMixin):
    """Cần allocation (self.allocation) nên kế thừa AllocationMixin."""

    def _create_po_commitment(self) -> "POCommitmentMixin":
        self._get_allocation()

        data = self.src

        template_po_commitment = {
            column_po_commitment[0]: data["SO"],
            column_po_commitment[1]: data["SO"],
            column_po_commitment[2]: data["GOLD CODE"],
            column_po_commitment[3]: data["LV"],
            column_po_commitment[4]: data["LU"],
            column_po_commitment[5]: data["PURCHASE NETWORK EXPANDED"],
            column_po_commitment[7]: "2",
            column_po_commitment[8]: data["SUPPLIER CODE"],
        }

        template_po_commitment = pd.DataFrame(template_po_commitment)

        template_po_commitment["SITE"] = template_po_commitment["SITE"].str.split(";")

        template_po_commitment = template_po_commitment.explode("SITE")

        template_po_commitment["KEY"] = (
            template_po_commitment["GOLD CODE"].astype(str)
            + "-"
            + template_po_commitment["LV"].astype(str)
            + "-"
            + template_po_commitment["SUPPLIER"].astype(str)
            + "-"
            + template_po_commitment["SITE"]
        )

        template_po_commitment = {
            **template_po_commitment,
            column_po_commitment[6]: template_po_commitment["KEY"].map(self.allocation),
        }

        template_po_commitment = pd.DataFrame(template_po_commitment)

        template_po_commitment = template_po_commitment[column_po_commitment]

        template_po_commitment = self.fast_stage(template_po_commitment, have_no=True)

        self.template_po_commitment = template_po_commitment

        return self


class SupplierScheduleMixin(CalendarMixin):
    def _create_supplier_schedule(self) -> "SupplierScheduleMixin":
        data = self.src

        template_supplier_schedule = {
            column_supplier_schedule[0]: data["SO"],
            column_supplier_schedule[1]: data["PURCHASE NETWORK EXPANDED"],
            column_supplier_schedule[2]: data["SUPPLIER CODE"],
            column_supplier_schedule[3]: "1",
            column_supplier_schedule[4]: data["COMMERCIAL CONTRACT"],
            column_supplier_schedule[5]: "",
            column_supplier_schedule[7]: "0000",
            column_supplier_schedule[9]: "2359",
            column_supplier_schedule[10]: "1",
            column_supplier_schedule[11]: "1",
            column_supplier_schedule[12]: "",
            "%DELI": data[["% DELIVERY 1", "% DELIVERY 2", "% DELIVERY 3"]].values.tolist(),
            "DELIVERY TYPE": data["DELIVERY TYPE"],
        }

        template_supplier_schedule = pd.DataFrame(template_supplier_schedule)

        template_supplier_schedule[["ORDER DATE", "DELIVERY DATE"]] = (
            template_supplier_schedule.apply(self._get_calendar, axis=1)
        )

        template_supplier_schedule = template_supplier_schedule.explode(
            ["%DELI", "ORDER DATE", "DELIVERY DATE"]
        )

        template_supplier_schedule["SITE"] = template_supplier_schedule["SITE"].str.split(";")

        template_supplier_schedule = template_supplier_schedule.explode("SITE")

        template_supplier_schedule["CONTRACT"] = template_supplier_schedule.apply(
            self.get_contract, axis=1
        )

        template_supplier_schedule = template_supplier_schedule[column_supplier_schedule]

        template_supplier_schedule = self.fast_stage(template_supplier_schedule, have_no=True)

        self.template_supplier_schedule = template_supplier_schedule

        return self


class AddAttributeMarketingMixin(AttributeMapMixin):
    def _create_add_attribute_marketing(self) -> "AddAttributeMarketingMixin":
        data = self.src

        template_add_attribute_marketing = {
            column_add_attribute_marketing[0]: "1",
            column_add_attribute_marketing[1]: data["SO"],
            column_add_attribute_marketing[2]: data["GOLD CODE"],
            column_add_attribute_marketing[3]: data["LV"],
            column_add_attribute_marketing[4]: data["LU"],
            column_add_attribute_marketing[5]: data["ATTRIBUTE MARKETING"],
            column_add_attribute_marketing[6]: data["FREE PRODUCT"],
        }

        template_add_attribute_marketing = pd.DataFrame(template_add_attribute_marketing)

        template_add_attribute_marketing["MEDIUM"] = template_add_attribute_marketing[
            "MEDIUM"
        ].map(self.attribute_map)

        template_add_attribute_marketing = template_add_attribute_marketing[
            column_add_attribute_marketing
        ]

        template_add_attribute_marketing = self.fast_stage(
            template_add_attribute_marketing, have_no=True
        )

        self.template_add_attribute_marketing = template_add_attribute_marketing

        return self

class Template_Mapping(
    BaseTemplate,
    CheckOAMixin,
    PromotionPlanMixin,
    UpdateSOMixin,
    MissingOUMixin,
    SOCalendarMixin,
    PurchaseMixin,
    POCommitmentMixin,
    SupplierScheduleMixin,
    AddAttributeMarketingMixin,
):
    pass


class DiscountTypeMixin:
    _NUMBER_PATTERN = re.compile(r"\d+(?:\.\d+)?")
 
    @staticmethod
    def get_discount_type(value) -> Optional[str]:
        if pd.isna(value):
            return None
 
        text = str(value)
 
        if "+" in text:
            return "3"
        if "%" in text:
            return "1"
        return "2"
 
    @classmethod
    def is_zero_discount(cls, value) -> bool:
        if pd.isna(value):
            return False
 
        text = str(value).strip()
        if text == "":
            return False
 
        numbers = cls._NUMBER_PATTERN.findall(text)
        if not numbers:
            return False
 
        return all(float(n) == 0 for n in numbers)
 
class Discount(ContractMixin, StageMixin, DiscountTypeMixin):
    def __init__(self, etl: "Template_ETL", username: str = "user"):
        self.src = etl.src
        self.etl = etl
        self.nw = etl.dict_network
        self.username = username
 
        self.template_ag_raw: Optional[pd.DataFrame] = None

        self.template_ag: Optional[pd.DataFrame] = None

        self.report_err: Optional[pd.DataFrame] = None
        
        self.template_dc_free: Optional[pd.DataFrame] = None
        self.template_dc_money: Optional[pd.DataFrame] = None
        
        self.template_de: Optional[pd.DataFrame] = None
 
    def _create_ag_raw(self) -> "Discount":
        data = self.src
        etl = self.etl
 
        today = pd.Timestamp.today().strftime("%d.%m")
 
        template_ag_raw = {
            column_ag[0]: "0",
            column_ag[1]: data["STRUCTURE"].str[:4],
            column_ag[2]: data["PURCHASE NETWORK EXPANDED"],
            column_ag[3]: data["SUPPLIER CODE"],
            column_ag[4]: data["COMMERCIAL CONTRACT"],
            column_ag[6]: "",
            column_ag[7]: f"{etl.cata}D.GP{etl.dept}-{self.username}({today})",
            column_ag[8]: data["PP START DATE"].min().strftime("%d/%m/%Y"),
            column_ag[9]: data["PP END DATE"].max().strftime("%d/%m/%Y"),
            column_ag[10]: data["GOLD CODE"],
            column_ag[11]: data["LV"],
            column_ag[12]: data["PP START DATE"].min().strftime("%d/%m/%Y"),
            column_ag[13]: data["PP END DATE"].max().strftime("%d/%m/%Y"),
            column_ag[14]: "0",
            column_ag[15]: "",
            "DISCOUNT VALUE": data["DISCOUNT (% OR VALUE)"],
            "RAW START DATE": data["PP START DATE"].dt.strftime("%d/%m/%Y"),
            "RAW END DATE": data["PP END DATE"].dt.strftime("%d/%m/%Y"),
        }
 
        template_ag_raw = pd.DataFrame(template_ag_raw)
 
        template_ag_raw["DISCOUNT TYPE"] = template_ag_raw["DISCOUNT VALUE"].apply(
            self.get_discount_type
        )
        
        template_ag_raw = template_ag_raw[
            ~template_ag_raw["DISCOUNT VALUE"].apply(self.is_zero_discount)
        ].reset_index(drop=True)
 
        template_ag_raw[column_ag[2]] = template_ag_raw[column_ag[2]].str.split(";")
        template_ag_raw = template_ag_raw.explode(column_ag[2])
 
        site_padded = template_ag_raw[column_ag[2]].astype(str).str.zfill(4)
        supplier = template_ag_raw[column_ag[3]].astype(str)
 
        key = supplier + site_padded
        sequence = key.groupby(key).cumcount() + 1
        sequence_str = sequence.apply(lambda x: f"{x:02d}")
 
        column_ag5_value = supplier + site_padded + sequence_str
 
        template_ag_raw = {**template_ag_raw, column_ag[5]: column_ag5_value}
 
        template_ag_raw = pd.DataFrame(template_ag_raw)
 
        template_ag_raw["CONTRACT"] = template_ag_raw.apply(self.get_contract, axis=1)
 
        template_ag_raw = self.fast_stage(template_ag_raw, have_no=True)
 
        self.template_ag_raw = template_ag_raw
 
        return self

    def _create_ag(self) -> "Discount":
        data = self.template_ag_raw

        template_ag = data[column_ag]

        template_ag = self.fast_stage(template_ag, have_no=True)
 
        self.template_ag = template_ag
 
        return self

    def _update(self, path_report_ag) -> "Discount":
        data = self.template_ag_raw
        
        report = pd.read_excel(
            path_report_ag,
            dtype=str
        )

        self.report_err = report.loc[~report["ERRORMESS"].isna()]
        report = report.loc[report["ERRORMESS"].isna()]

        group_columns = [
            'ACTION', 'DEPARTMENT', 'SITE', 'SUPPLIER', 'CONTRACT', 'AG NO', 
            'AG DESCRIPTION', 'AG START DATE', 'AG END DATE', 'GOLD CODE', 
            'LV', 'ARTICLE START DATE', 'ARTICLE END DATE'
        ]
        
        group_report_columns = [
            'ACTION', 'DEPT', 'SITE', 'SUPPLIER_CODE', 'COMERCIAL_CONTRACT', 'AGNO1', 
            'AG_DESC', 'AG_START_DATE', 'AG_END_DATE', 'ARTICLE_CODE', 
            'LV', 'ARTICLE_START_DATE', 'ARTICLE_END_DATE'
        ]

        data["KEY"] = data[group_columns].astype(str).agg("-".join, axis=1)
        report["KEY"] = report[group_report_columns].astype(str).agg("-".join, axis=1)

        report_key = dict(zip(report["KEY"], report["AG_CODE"]))

        data["AG CODE"] = data["KEY"].map(report_key)

        self.template_ag_raw = data.loc[
            data["AG CODE"].fillna("").str.strip().ne("")
        ]

        return self

    def _create_dc(self) -> "Discount":
        ag_raw = self.template_ag_raw
        
        mask_free = ag_raw["DISCOUNT TYPE"] == "3"

        template_dc_free = {
            column_dc[0]:"0",
            column_dc[1]:ag_raw.loc[mask_free, "SITE"],
            column_dc[2]:ag_raw.loc[mask_free, "SUPPLIER"],
            column_dc[3]:ag_raw.loc[mask_free, "CONTRACT"],
            column_dc[4]:ag_raw.loc[mask_free, "AG CODE"],
            column_dc[5]:ag_raw.loc[mask_free, "AG DESCRIPTION"],
            column_dc[6]:ag_raw.loc[mask_free, "AG CODE"],
            column_dc[7]:"501",
            column_dc[8]:"1",
            column_dc[9]:"2",
            column_dc[10]:"0",
            column_dc[11]:"20",
            column_dc[12]:"20",
            column_dc[13]:"ARTICLE START DATE",
            column_dc[14]:"ARTICLE END DATE",
            column_dc[15]:"3",
            column_dc[16]:"0",
            column_dc[17]:"1",
            column_dc[18]:ag_raw.loc[mask_free, "GOLD CODE"],
            column_dc[19]:ag_raw.loc[mask_free, "LV"],
            column_dc[24]:ag_raw.loc[mask_free, "RAW START DATE"],
            column_dc[25]:ag_raw.loc[mask_free, "RAW END DATE"],
            "VALUE FOR FREE":ag_raw.loc[mask_free, "DISCOUNT VALUE"],
            column_dc[26]:""
        }
        
        template_dc_free = pd.DataFrame(template_dc_free)

        if not template_dc_free.empty:
        
            template_dc_free[[column_dc[20], column_dc[22]]] = template_dc_free["VALUE FOR FREE"].str.split("+", expand=True)
            
            template_dc_free[column_dc[21]] = template_dc_free[column_dc[20]].map(
                lambda x: "41" if pd.notna(x) and ("TH" in x or "T" in x) else "1"
            )
            
            template_dc_free[column_dc[23]] = template_dc_free[column_dc[22]].map(
                lambda x: "41" if pd.notna(x) and ("TH" in x or "T" in x) else "1"
            )
            
            template_dc_free[column_dc[20]] = (
                template_dc_free[column_dc[20]]
                .str.replace("TH", "", regex=False)
                .str.replace("T", "", regex=False)
                .str.strip()
            )
            
            template_dc_free[column_dc[22]] = (
                template_dc_free[column_dc[22]]
                .str.replace("TH", "", regex=False)
                .str.replace("T", "", regex=False)
                .str.strip()
            )
    
            template_dc_free = template_dc_free[column_dc]
            template_dc_free = self.fast_stage(template_dc_free, have_no=True)
            self.template_dc_free = template_dc_free

        template_dc_money = {
            column_dc[0]:"0",
            column_dc[1]:ag_raw.loc[~mask_free, "SITE"],
            column_dc[2]:ag_raw.loc[~mask_free, "SUPPLIER"],
            column_dc[3]:ag_raw.loc[~mask_free, "CONTRACT"],
            column_dc[4]:ag_raw.loc[~mask_free, "AG CODE"],
            column_dc[5]:ag_raw.loc[~mask_free, "AG DESCRIPTION"],
            column_dc[6]:ag_raw.loc[~mask_free, "AG CODE"],
            column_dc[7]:"501",
            column_dc[8]:"1",
            column_dc[9]:"2",
            column_dc[10]:"0",
            column_dc[11]:"20",
            column_dc[12]:"20",
            column_dc[13]:ag_raw.loc[~mask_free, "ARTICLE START DATE"],
            column_dc[14]:ag_raw.loc[~mask_free, "ARTICLE END DATE"],
            column_dc[15]:ag_raw.loc[~mask_free, "DISCOUNT TYPE"],
            column_dc[16]:"0",
            column_dc[17]:"1",
            column_dc[18]:"",
            column_dc[19]:"",
            column_dc[20]:"",
            column_dc[21]:"",
            column_dc[22]:"",
            column_dc[23]:"",
            column_dc[24]:ag_raw.loc[~mask_free, "ARTICLE START DATE"],
            column_dc[25]:ag_raw.loc[~mask_free, "ARTICLE END DATE"],
            column_dc[26]:""
        }

        template_dc_money = pd.DataFrame(template_dc_money)
        
        if not template_dc_money.empty:
            
            template_dc_money = template_dc_money[column_dc]
    
            template_dc_money = self.fast_stage(template_dc_money, have_no=True)
     
            self.template_dc_money = template_dc_money
 
        return self

    def _create_de(self) -> "Discount":
        raw = self.template_ag_raw
        
        raw = raw[raw["DISCOUNT TYPE"] != "3"]
        
        template_de = {
            column_de[0]:"0",
            column_de[1]:raw["SITE"],
            column_de[2]:raw["AG CODE"],
            column_de[3]:raw["AG CODE"],
            column_de[4]:raw["GOLD CODE"],
            column_de[5]:raw["LV"],
            column_de[6]:raw["DISCOUNT VALUE"],
            column_de[7]:raw["RAW START DATE"],
            column_de[8]:raw["RAW END DATE"],
            column_de[9]:"",
            column_de[10]:"",
        }
        
        template_de = pd.DataFrame(template_de)
        
        template_de["VALUE ON INVOICE"] = template_de["VALUE ON INVOICE"].str.replace("%","").str.strip()

        template_de = template_de[column_de]

        template_de = self.fast_stage(template_de, have_no=True)
 
        self.template_de = template_de
 
        return self

class SalePrice(StageMixin):
    def __init__(self, etl: "Template_ETL"):
        self.src = etl.src_listoff
        self.etl = etl
 
        self.template_sp: Optional[pd.DataFrame] = None
        self.template_attr: Optional[pd.DataFrame] = None

    def _create_sp(self) -> "SalePrice":
        data = self.src.copy()
        template_sp = {
            column_sale[0] : "0",
            column_sale[1] : data["GOLD CODE"],
            column_sale[2] : data["SV"],
            column_sale[3] : data["PROMOTION SALE PRICE"],
            column_sale[4] : data["PRICELIST"],
            "PRICELIST CODE" : data["PRICELIST CODE"],
            column_sale[5] : data["SP START DATE"].dt.strftime("%d/%m/%Y"),
            column_sale[6] : data["SP END DATE"].dt.strftime("%d/%m/%Y"),
            column_sale[7] : data["SALE VAT"].map(VAT),
            column_sale[8] : ""
        }

        template_sp = pd.DataFrame(template_sp)
        template_sp["PRICELIST"] = template_sp["PRICELIST"].str.split(";")
        template_sp = template_sp.explode("PRICELIST")
        def _get_plcode(x):
            PRICELIST = x["PRICELIST"]
            CODE = x["PRICELIST CODE"]
        
            if len(PRICELIST) == 3:
                return f"{CODE}0{PRICELIST}"
            if len(PRICELIST) == 4:
                return f"{CODE}{PRICELIST}"
        
        template_sp["PRICELIST"] = template_sp.apply(_get_plcode, axis=1)
        template_sp = template_sp[column_sale]

        template_sp = self.fast_stage(template_sp, have_no=True)

        self.template_sp = template_sp

        return self

In [6]:
mp = Template_Mapping(etl)

In [7]:
sp = SalePrice(etl)

In [8]:
sp._create_sp()

In [7]:
discount = Discount(etl, "tri")

In [183]:
discount._create_ag_raw()
discount._create_ag()
discount._update(settings.path_report_ag)
discount._create_dc()

In [10]:
mp._create_purchase()

In [64]:
import pandas as pd
import re
from typing import Optional, List

In [65]:
dt = pd.read_excel(r"C:\Users\kt20361464\Downloads\ihi\C614\410 - TỔNG HỢP THÔNG TIN final  - C614.xlsx", dtype=str, sheet_name='C614')

In [66]:
required_headers = {
    'GOLD CODE',
    'SV',
    'START DATE',
    'END DATE',
    'REGION (NORTH/SOUTH/CENTER/ALL)',
    'PAGE',
    'THEMATIC',
    'POSITION',
}

In [67]:
def scan_header(df):

    for idx in range(min(10, len(df))):

        actual_headers = set(
            str(x).strip()
            for x in df.iloc[idx]
            if str(x).strip() != ''
        )

        missing_headers = required_headers - actual_headers

        if not missing_headers:
            return {
                'valid': True,
                'header_row': idx,
                'missing': [],
                'extra': sorted(actual_headers - required_headers),
            }

    return {
        'valid': False,
        'header_row': None,
        'missing': sorted(required_headers),
        'extra': [],
    }

In [68]:
result = scan_header(dt)

if not result['valid']:
    print("Missing headers:", result['missing'])
else:
    print("Header OK")

Header OK


In [69]:
dt = pd.read_excel(r"C:\Users\kt20361464\Downloads\ihi\C614\410 - TỔNG HỢP THÔNG TIN final  - C614.xlsx", dtype=str, sheet_name='C614', header=result['header_row']+1)

In [70]:
dt = dt.iloc[1:].reset_index(drop=True)

In [71]:
def _check_required_data(
    data: Optional[pd.DataFrame],
    required: List[str]
) -> Optional[pd.DataFrame]:
    if data is None or data.empty:
        return data

    empty_cols = [
        col
        for col in required
        if data[col].replace("", pd.NA).isna().any()
    ]

    if empty_cols:
        raise ValueError(f"Required columns contain empty values: {', '.join(empty_cols)}")

    return data

In [82]:
_check_required_data(dt, required_headers);

In [73]:
dt = dt[list(required_headers)]

In [78]:
CATEGORY_RULES: Optional[List] = [
    (
        re.compile(r"(?i)\b(front\s*page|back\s*page|unbeat)\b"),
        "HEROP",
    ),
    (
        re.compile(
            r"(?i)\b(cata|catalog(?:ue)?|fair|member\s*price|banner|exclusive\s*pack|family|other|normal|the\s*1)\b"
        ),
        "CATAP",
    ),
    (
        re.compile(r"(?i)\b(comple(?:mentary)?|comple)\b"),
        "COMPLEP",
    ),
    (
        re.compile(r"(?i)\bbuy\s*more\s*save\s*more\b"),
        "STARP",
    ),
]

def attribute_map(text: str) -> str:
    if not text:
        return text

    text = str(text).strip()

    for pattern, value in CATEGORY_RULES:
        if pattern.search(text):
            return value

    return text

In [85]:
dt["CODE"] = (
    dt["POSITION"].astype(str).str.strip()
    + ".P."
    + dt["PAGE"].astype(str).str.strip()
    + "."
    + dt["REGION (NORTH/SOUTH/CENTER/ALL)"].astype(str).str.strip()
)

In [86]:
dt

,GOLD CODE,REGION (NORTH/SOUTH/CENTER/ALL),END DATE,PAGE,POSITION,START DATE,THEMATIC,SV,CLASS,CODE
0,00641148,8200,15/07/2026,15,NORMAL,02/07/2026,NORMAL PAGE,1,CATAP,NORMAL.P.15.8200
1,00641147,8200,15/07/2026,15,NORMAL,02/07/2026,NORMAL PAGE,1,CATAP,NORMAL.P.15.8200
2,00038342,8200,15/07/2026,15,NORMAL,02/07/2026,THE 1,0,CATAP,NORMAL.P.15.8200
3,01396210,8200,15/07/2026,15,NORMAL,02/07/2026,THE 1,1,CATAP,NORMAL.P.15.8200
4,01150542,8200,15/07/2026,15,NORMAL,02/07/2026,THE 1,1,CATAP,NORMAL.P.15.8200
...,...,...,...,...,...,...,...,...,...,...
108,00987388,8200,15/07/2026,2,OTHER,02/07/2026,SHOCK DEALS,1,CATAP,OTHER.P.2.8200
109,00987390,8200,15/07/2026,2,OTHER,02/07/2026,SHOCK DEALS,1,CATAP,OTHER.P.2.8200
110,00987389,8200,15/07/2026,2,OTHER,02/07/2026,SHOCK DEALS,1,CATAP,OTHER.P.2.8200
111,02107827,8200,15/07/2026,14,NORMAL,02/07/2026,NORMAL PAGE,1,CATAP,NORMAL.P.14.8200
